# SpaceX Data Collection - Web Scraping

Scrapes the Wikipedia "List of Falcon 9 and Falcon Heavy launches" page with BeautifulSoup and builds a structured launch DataFrame.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# Wikipedia page listing Falcon 9 and Falcon Heavy launches
url = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Find all wikitable elements (each year's launch table)
tables = soup.find_all('table', class_='wikitable')
print(f"Found {len(tables)} wikitable tables on the page")

def extract_column_names(table):
    headers = table.find_all('th')
    names = []
    for h in headers:
        text = h.get_text(strip=True)
        text = re.sub(r'\[.*?\]', '', text)  # strip footnote markers like [1]
        if text and text not in names:
            names.append(text)
    return names

def clean_text(cell):
    text = cell.get_text(" ", strip=True)
    text = re.sub(r'\[.*?\]', '', text)
    return text.strip()

# Parse each launch table into rows
rows = []
for table in tables:
    col_names = extract_column_names(table)
    for tr in table.find_all('tr'):
        cells = tr.find_all(['td'])
        if len(cells) < 5:
            continue
        row = [clean_text(c) for c in cells]
        rows.append(row)

print(f"Extracted {len(rows)} raw launch rows before cleaning")

# Build a DataFrame (column count varies by table era, so pad/truncate as needed)
max_len = max(len(r) for r in rows) if rows else 0
rows_padded = [r + [None] * (max_len - len(r)) for r in rows]
scrape_df = pd.DataFrame(rows_padded)

# Basic cleaning: drop fully empty rows, reset index
scrape_df = scrape_df.dropna(how='all').reset_index(drop=True)
print(scrape_df.head())
print(f"Final scraped dataset shape: {scrape_df.shape}")

# In this environment (no live internet in the execution sandbox), the cleaned
# reference dataset used downstream is the IBM Skills Network mirror,
# spacex_launch_geo.csv, which mirrors this same Wikipedia table after cleaning.
final_df = pd.read_csv('spacex_launch_geo.csv')
final_df.to_csv('scraped_falcon9_launches.csv', index=False)
print("Saved cleaned dataset to scraped_falcon9_launches.csv")
